In [4]:
"""
EXTERNAL VALIDATION: Train on Camel/Hadoop, test on Kafka/Tika (real,
out-of-sample generalization test)
================================================================================
This directly answers the reviewer's suggestion: instead of testing each
project independently (RQ1's original design), this trains a real model on
Camel/Hadoop only, then tests it on Kafka and Tika, projects the model has
never seen. This is a genuinely different, stronger claim than "each project
independently beats its own baseline" -- it tests whether patterns learned
on one real codebase transfer to different real codebases.

Uses only real, already-available data -- no new extraction needed.
"""
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

FEATURES = ["loc", "cyclomatic_complexity", "num_functions", "num_files_changed"]


def load_and_label(path, target_col):
    df = pd.read_csv(path)
    df["era_binary"] = (df["era"] == "ai_era").astype(int)
    df["label"] = df[target_col]
    return df


def run_external_validation():
    train_df = load_and_label("rq1_refined_with_real_issue_type.csv", "defect_prone_strict")
    print(f"Real training data (Camel/Hadoop): N={len(train_df)}")

    df = pd.read_csv("rq1_refined_with_real_issue_type.csv")
    print(df["era"].value_counts())

    X_train = train_df[FEATURES + ["era_binary"]].values
    y_train = train_df["label"].values

    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                  random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    # Real, in-sample baseline for reference (same project, held-out split
    # would normally be used, but here we report full-train accuracy as a
    # ceiling reference, not a real test metric)
    train_pred = rf.predict(X_train)
    print(f"  Real in-sample (training) accuracy: {accuracy_score(y_train, train_pred)*100:.1f}% "
          f"(reference ceiling only, not a real test)")

    print("\n=== REAL EXTERNAL VALIDATION: testing on unseen real projects ===")
    for name, path, target in [("Kafka", "kafka_real_mined_dataset.csv", "defect_prone"),
                                  ("Tika", "tika_real_mined_dataset.csv", "defect_prone")]:
        test_df = load_and_label(path, target)
        X_test = test_df[FEATURES + ["era_binary"]].values
        y_test = test_df["label"].values

        pred = rf.predict(X_test)
        proba = rf.predict_proba(X_test)[:, 1]
        maj_pred = np.ones_like(y_test) if y_test.mean() > 0.5 else np.zeros_like(y_test)

        print(f"\n{name} (real N={len(test_df)}, model never trained on this project):")
        print(f"  Real accuracy:  {accuracy_score(y_test, pred)*100:.1f}%")
        print(f"  Real precision: {precision_score(y_test, pred, zero_division=0):.3f}")
        print(f"  Real recall:    {recall_score(y_test, pred, zero_division=0):.3f}")
        print(f"  Real F1:        {f1_score(y_test, pred, zero_division=0):.3f}")
        try:
            print(f"  Real AUC:       {roc_auc_score(y_test, proba):.3f}")
        except ValueError:
            print(f"  Real AUC:       undefined (single class in test set)")
        print(f"  Real majority baseline: {accuracy_score(y_test, maj_pred)*100:.1f}%")


if __name__ == "__main__":
    run_external_validation()

Real training data (Camel/Hadoop): N=18636
era
pre_ai    15250
ai_era     3386
Name: count, dtype: int64
  Real in-sample (training) accuracy: 66.3% (reference ceiling only, not a real test)

=== REAL EXTERNAL VALIDATION: testing on unseen real projects ===

Kafka (real N=433, model never trained on this project):
  Real accuracy:  62.6%
  Real precision: 0.576
  Real recall:    0.611
  Real F1:        0.593
  Real AUC:       0.665
  Real majority baseline: 55.4%

Tika (real N=395, model never trained on this project):
  Real accuracy:  59.7%
  Real precision: 0.400
  Real recall:    0.718
  Real F1:        0.514
  Real AUC:       0.685
  Real majority baseline: 70.4%
